# Octavo

> Build nbdev documentation with Octavo
- order: 21

In [ ]:
#| default_exp octavo

Install `nbdev[octavo]` to enable this integration. `nbdev-docs` selects Octavo when `octavo.yml` exists in the configured notebooks directory; otherwise it uses Quarto. The Quarto path does not import Octavo. Octavo builds the whole site; non-default Quarto file filters are rejected, and `n_workers` only controls Quarto builds.

Run `nbdev-export` before building so generated modules and symbol indexes are current. Migration is a source conversion, not a way to render unchanged notebooks with either backend. README and CONTRIBUTING use Octavo for opted-in projects. `nbdev-prepare` retains export, test, and clean, skips Quarto configuration generation for Octavo projects, and dispatches README and CONTRIBUTING to the selected backend. Preview and the migration command are not yet connected to this integration.

In [ ]:
#| export
from contextlib import redirect_stdout
from io import StringIO
from fastcore.utils import *
import yaml
from nbdev.config import get_config, import_obj, _load_toml
from nbdev.doclinks import NbdevLookup
from nbdev.process import NBProcessor
from nbdev.processors import FilterDefaults, add_links
from nbdev.serve import _is_qpy
from octavo.site import Site
from octavo.pages import Page, GeneratedFile, scan, page_nm, hidden, ipynb2md
from octavo.core import md2doc
from mdhtml import md2gfm
from nbdev.quarto import _doc_mtime_not_older

In [ ]:
#| hide
import tempfile
from copy import deepcopy
from fastcore.test import *
from fastcore.nbio import new_nb, mk_cell, write_nb
from nbdev.quarto import nbdev_docs

## Notebook processing

`OctavoProc` retains custom `doc_procs` and `skip_procs` from nbdev. It preserves rendering directives and exposes generated YAML frontmatter as Markdown. Generated project-symbol links are local; authored production URLs are unchanged.

In [ ]:
#| export
class OctavoProc(FilterDefaults):
    "Process nbdev notebooks for Octavo without changing Quarto defaults"
    def __init__(self, local_lib=None, strip_libs=None): self.nl = NbdevLookup(strip_libs=strip_libs or local_lib, local_lib=local_lib)
    def add_links(self, cell): return add_links(cell, nl=self.nl)
    def procs(self): return L(self.add_links if o is add_links else o for o in super().procs())
    def nb_proc(self, nb): return NBProcessor(nb=nb, procs=self.procs(), rm_directives=False)

    def __call__(self, nb):
        super().__call__(nb)
        if getattr(nb, 'frontmatter_', None) and nb.cells[0].cell_type=='raw': nb.cells[0].cell_type = 'markdown'

In [ ]:
source = new_nb([mk_cell('# Example\n\n> A notebook with rendering directives', 'markdown'),
                 mk_cell('Use `ModuleMaker`. See [production docs](https://nbdev.fast.ai/api/maker.html#modulemaker).', 'markdown'),
                 mk_cell('#| echo: false\n1 + 1')])
source.path_ = Path('example.ipynb')
nb = deepcopy(source)
OctavoProc(local_lib='nbdev')(nb)
test_eq(nb.cells[0].cell_type, 'markdown')
text = '\n'.join(c.source for c in nb.cells)
assert '[`ModuleMaker`](/api/maker.html#modulemaker)' in text
assert '[production docs](https://nbdev.fast.ai/api/maker.html#modulemaker)' in text
assert '#| echo: false' in text
quarto_nb = deepcopy(source)
FilterDefaults()(quarto_nb)
test_eq(quarto_nb.cells[0].cell_type, 'raw')
assert '#| echo: false' in '\n'.join(c.source for c in quarto_nb.cells)
nb.cells[0].source

A distribution name need not match its import package or nbdev entry-point name. Site builds use the project's declared nbdev entry points to select local symbol indexes; README generation keeps published symbol URLs.

In [ ]:
#| export
def local_libraries(cfg):
    "Names of the project's declared nbdev symbol indexes"
    proj = _load_toml(cfg.config_file).get('project', {})
    return tuple(proj.get('entry-points', {}).get('nbdev', {}))

In [ ]:
with tempfile.TemporaryDirectory() as d:
    project = Path(d)
    (project/'pyproject.toml').write_text('[project]\nname = "different-distribution"\n[project.entry-points.nbdev]\nnbdev = "nbdev._modidx:d"\n[tool.nbdev]\n')
    libs = local_libraries(get_config(project))
    test_eq(libs, ('nbdev',))
    lookup = NbdevLookup(local_lib=libs)
    assert lookup.doc('nbdev.maker.ModuleMaker').startswith('/api/maker.html#')
    assert NbdevLookup().doc('nbdev.maker.ModuleMaker').startswith('https://')
    nb = deepcopy(source)
    with working_directory(project): OctavoProc(local_lib=libs)(nb)
    assert '[`ModuleMaker`](/api/maker.html#modulemaker)' in '\n'.join(c.source for c in nb.cells)

## Render scripts

A *render script* is a Python file with a double suffix, such as `page.md.py` or `drawing.svg.py`. nbdev executes it and saves stdout to the rendered Octavo site without the final `.py` suffix. A `.md.py` file may declare frontmatter in its module docstring; every other generated file contains exactly what its script prints.

Octavo never executes anything: nbdev's `script_pages` hands it finished `Page`s and `GeneratedFile`s. A script that fails to run fails the build, since a half-built site is worse than a clear error. Renderscript sources such as `fname.ext.py` are not sent to Octavo, while ordinary single-suffix `.py`, like any non-page sourcefile, files are published as assets.

In [ ]:
#| export
def render_script(path):
    "Captured stdout from a double-suffix render script; prepend optional `.md.py` frontmatter"
    path = Path(path)
    if path.suffix!='.py' or len(path.suffixes)<2: return
    fm = _is_qpy(path) if path.name.endswith('.md.py') else None
    f = StringIO()
    with redirect_stdout(f): exec(compile(path.read_text(encoding='utf-8'), str(path), 'exec'), {})
    return (f'---\n{fm}\n---\n\n' if fm else '') + f.getvalue()

In [ ]:
#| export
def script_pages(root):
    "Render scripts under `root`: `*.md.py` as `Page`s, anything else as `GeneratedFile`s"
    root,pages,files = Path(root),L(),L()
    for o in sorted(root.rglob('*.py')):
        rel = o.relative_to(root)
        if hidden(rel) or (txt:=render_script(o)) is None: continue
        dest = rel.with_suffix('')
        if dest.suffix=='.md': pages.append(Page(str(rel), txt, nm=page_nm(dest)))
        else: files.append(GeneratedFile(str(rel), str(dest), txt))
    return pages,files

In [ ]:
with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    (d/'01_intro.md.py').write_text('"""---\ntitle: Hi\n---"""\nprint("# Generated")\nprint("body")')
    (d/'images').mkdir(); (d/'images/dot.svg.py').write_text('print("<svg/>")')
    (d/'plain.md.py').write_text('print("# No frontmatter")')
    (d/'ordinary.py').write_text('x = 1')
    (d/'_skip').mkdir(); (d/'_skip/no.md.py').write_text('"""---\n---"""\nprint("no")')
    pages,files = script_pages(d)
    test_eq([p.dest for p in pages], ['intro.html','plain.html'])
    test_eq(pages[0].text, '---\ntitle: Hi\n---\n\n# Generated\nbody\n')
    test_eq(pages[0].meta['title'], 'Hi')
    test_eq(pages[1].text, '# No frontmatter\n')
    test_eq([(f.src,f.dest,f.content) for f in files], [('images/dot.svg.py','images/dot.svg','<svg/>\n')])
    test_is(render_script(d/'ordinary.py'), None)
    (d/'bad.md.py').write_text('1/0')
    test_fail(lambda: script_pages(d), contains='division')

## Post-build callback

Set `docs_post_build` to an importable `module:function` in `pyproject.toml`:

```toml
[tool.nbdev]
docs_post_build = "my_project.docbuild:build_extras"
```

After a successful site build, nbdev calls the function once with `(out, cfg)`. `out` is the absolute directory containing the completed site. `cfg` is the nbdev project configuration. The callback writes additional files into `out`; its return value is ignored. Callback errors fail the build. Without this setting, no callback runs.

Author the callback in a notebook and export it before building the docs, as with custom processors. Its dependencies belong to the project's documentation setup.

## Build configuration

`octavo.yml` must exist and contain a YAML mapping. An empty file uses defaults. Validate it before executing render scripts or processing notebooks.

In [ ]:
#| export
def octavo_config(root):
    "Read an existing octavo.yml mapping before running project code"
    root = Path(root)
    if not (root/'octavo.yml').is_file(): raise FileNotFoundError(root/'octavo.yml')
    settings = yaml.safe_load((root/'octavo.yml').read_text())
    if settings is None: return {}
    if not isinstance(settings, dict): raise ValueError(f'{root / "octavo.yml"} must contain a YAML mapping')
    return settings

In [ ]:
with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    test_fail(lambda: octavo_config(root), contains='octavo.yml')
    for text in ('- not a mapping', '[]', 'false', '0'):
        (root/'octavo.yml').write_text(text)
        test_fail(lambda: octavo_config(root), contains='mapping')
    (root/'octavo.yml').write_text('')
    test_eq(octavo_config(root), {})

The site build gives nbdev’s symbol linker the current project identity. Generated project references become site-root-relative document addresses; dependency references and explicitly authored absolute links remain absolute. Octavo resolves local document addresses for each output format.

In [ ]:
#| export
def octavo_docs(
    path:str=None, # Path to the project or its notebooks
):
    "Create Octavo docs and run the optional docs_post_build callback"
    cfg = get_config(path)
    settings = octavo_config(cfg.nbs_path)
    with working_directory(cfg.config_path):
        pages,files = script_pages(cfg.nbs_path)
        docs = scan(cfg.nbs_path, nbproc=OctavoProc(local_lib=local_libraries(cfg))) + pages
        site = Site(docs, root=cfg.nbs_path, files=files, **settings)
        if (miss := site.build()): print(miss)
        if (post := cfg.get('docs_post_build')):
            out = Path(site.settings['output']).expanduser()
            if not out.is_absolute(): out = cfg.nbs_path/out
            import_obj(post)(out.resolve(), cfg)

## Build an opted-in project

`nbdev-docs` reads `octavo.yml` and processes notebooks and render scripts together. It does not migrate sources or change `_quarto.yml`. This temporary project builds linked pages and generated assets, then demonstrates that invalid configuration is rejected before executing a render script.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    project = Path(d)
    nbs = project/'nbs'
    nbs.mkdir()
    (project/'pyproject.toml').write_text('[project]\nname = "demo"\n[tool.nbdev]\nnbs_path = "nbs"\nskip_procs = "insert_warning"\n')
    (nbs/'octavo.yml').write_text('title: Demo\noutput: ../_octavo\nsite_url: https://example.com\n')
    (nbs/'_quarto.yml').write_text('project:\n  type: website\n')
    (nbs/'index.md').write_text('# Home\n\nRead the [guide](guide.ipynb).')
    guide = new_nb([mk_cell('# Guide\n\n> A notebook page', 'markdown'), mk_cell('Notebook content.', 'markdown')])
    write_nb(guide, nbs/'guide.ipynb')
    (nbs/'generated.md.py').write_text('print("# Generated page")')
    (nbs/'image.svg.py').write_text('print("<svg/>")')
    cwd = Path.cwd()
    with working_directory(project): run(r'nbdev-docs --file-re "\.(?:ipynb|qmd|html)$"')
    nbdev_docs.__wrapped__(project, file_glob=None)
    test_eq(Path.cwd(), cwd)
    out = project/'_octavo'
    assert 'guide.html' in (out/'index.html').read_text()
    assert 'Notebook content.' in (out/'guide.html').read_text()
    assert 'AUTOGENERATED' not in (out/'guide.md').read_text()
    assert (out/'generated.html').exists()
    test_eq((out/'image.svg').read_text(), '<svg/>\n')
    test_eq((nbs/'_quarto.yml').read_text(), 'project:\n  type: website\n')
    test_fail(lambda: nbdev_docs.__wrapped__(project, file_glob='*.ipynb'), contains='file_glob')
    (nbs/'octavo.yml').write_text('- invalid')
    (nbs/'generated.md.py').write_text('raise RuntimeError("script ran before validation")')
    test_fail(lambda: nbdev_docs.__wrapped__(project), contains='mapping')
    test_eq(Path.cwd(), cwd)

## Repository Markdown

`nbdev-readme` and `nbdev-contributing` render the configured notebooks to the project root without Quarto. They use stored outputs and do not execute ordinary code cells. nbdev processors still run, including custom processors and generated API documentation. Titles and descriptions become Markdown. Symbol links use published documentation URLs.

Embedded images are extracted into `README_files` or `CONTRIBUTING_files`. Authored relative destinations are unchanged, including reference-style links. Authors must choose paths suitable for the repository-root document. A missing source notebook is skipped; `chk_time=True` skips a document newer than its source.

In [ ]:
#| export
def octavo_readme(
    path:str=None, # Path to the project or its notebooks
    chk_time:bool=False, # Only build if the source is newer
    contributing:bool=False, # Render CONTRIBUTING instead of README
):
    "Render a repository Markdown document from a notebook's stored outputs"
    cfg = get_config(path)
    nbname = cfg.get('contributing_nb', 'contributing.ipynb') if contributing else cfg.readme_nb
    root = Path(path).absolute() if path else cfg.nbs_path
    src = root/nbname
    if not src.exists(): src = cfg.nbs_path/nbname
    if not src.exists(): return
    name = 'CONTRIBUTING' if contributing else 'README'
    dest = cfg.config_path/f'{name}.md'
    if chk_time and _doc_mtime_not_older(dest, src): return
    octavo_config(cfg.nbs_path)
    with working_directory(cfg.config_path):
        proc = OctavoProc(strip_libs=local_libraries(cfg))
        doc = md2doc(ipynb2md(src, nbproc=proc))
        head = f'# {doc.meta["title"]}\n\n' if doc.meta.get('title') else ''
        if desc:=doc.meta.get('description'): head += f'{desc}\n\n'
        md2gfm(head + doc.body, dest=dest, imgdir=cfg.config_path/f'{name}_files', raw=('md', 'html'))

In [ ]:
with tempfile.TemporaryDirectory() as d:
    project = Path(d)
    nbs = project/'nbs'
    nbs.mkdir()
    (project/'pyproject.toml').write_text('[project]\nname = "different-distribution"\n[project.entry-points.nbdev]\nnbdev = "nbdev._modidx:d"\n[tool.nbdev]\nnbs_path = "nbs"\ncontributing_nb = "help.ipynb"\n')
    (nbs/'octavo.yml').write_text('{}')
    png = 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAwMCAO+aX1sAAAAASUVORK5CYII='
    image = mk_cell('#| echo: false\nraise RuntimeError("must not execute")')
    image.outputs = [dict(output_type='display_data', metadata={}, data={'image/png': png})]
    book = new_nb([mk_cell('# Example\n\n> Repository document', 'markdown'),
                   mk_cell('Use `ModuleMaker`. [Guide](guide.md). [Reference][ref]\n\n[ref]: guide.md', 'markdown'),
                   image, mk_cell('#| include: false\n"excluded cell"')])
    write_nb(book, nbs/'index.ipynb')
    write_nb(book, nbs/'help.ipynb')
    cwd = Path.cwd()
    octavo_readme(project)
    with working_directory(project):
        run('nbdev-readme')
        run('nbdev-contributing')
    test_eq(Path.cwd(), cwd)
    for name in ('README', 'CONTRIBUTING'):
        text = (project/f'{name}.md').read_text()
        assert '# Example' in text and 'Repository document' in text
        assert 'https://nbdev.fast.ai/api/maker.html#modulemaker' in text
        assert '(guide.md)' in text and '[ref]: guide.md' in text
        assert 'must not execute' not in text and 'excluded cell' not in text
        assert 'data:image' not in text and f'{name}_files/' in text
        assert list((project/f'{name}_files').glob('*.png'))
    before = (project/'README.md').stat().st_mtime_ns
    octavo_readme(project, chk_time=True)
    test_eq((project/'README.md').stat().st_mtime_ns, before)
    assert not (nbs/'_quarto.yml').exists()
    (nbs/'help.ipynb').unlink()
    octavo_readme(project, contributing=True)